# 📘 Pandas: stack()/unstack() & MultiIndex クックブック
- 作成日: 2025-10-05 12:17
- コンセプト: **「こう書いたらこうなる」** を最短で体に染み込ませる実行用ノート
- 使い方: 上から順にセルを実行して「入力 → 出力」を観察してください。必要に応じて値を変えて振る舞いを確認しましょう。

---


In [1]:
# === セットアップ ===
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

print("pandas version:", pd.__version__)


pandas version: 2.2.3


## 0. まずは「縦⇄横」の最小例
「列⇄行」に変換する `stack()` / `unstack()` の感覚をつかむ最短の例です。


In [2]:
# --- 最小例：wide（横持ち）を tidy（縦持ち）へ ---
df = pd.DataFrame({
    "県": ["東京", "大阪"],
    "1月": [100, 200],
    "2月": [150, 250],
}).set_index("県")

print("▼元（横持ち）")
display(df)

print("\n▼stack(): 列（1月/2月）→ 行の第2階層へ")
s = df.stack()
display(s)  # Series（行MultiIndex）

print("\n▼unstack(): 行の第2階層 → 列へ戻す（逆操作）")
display(s.unstack())


▼元（横持ち）


,1月,2月
県,,
東京,100,150
大阪,200,250



▼stack(): 列（1月/2月）→ 行の第2階層へ


県     
東京  1月    100
    2月    150
大阪  1月    200
    2月    250
dtype: int64


▼unstack(): 行の第2階層 → 列へ戻す（逆操作）


,1月,2月
県,,
東京,100,150
大阪,200,250


## 1. MultiIndex（階層型インデックス）の基本
`set_index()` で複数列をインデックスにすると、行に階層ができます。`stack/unstack` はここを行き来します。


In [ ]:
base = pd.DataFrame({
    "県": ["東京","東京","大阪","大阪"],
    "店舗": ["A店","B店","A店","B店"],
    "売上": [100,150,200,180]
})
print("▼通常の表")
display(base)

m = base.set_index(["県","店舗"])
print("\n▼行に2階層：MultiIndex")
display(m)
print("index.names:", m.index.names)


▼通常の表


,県,店舗,売上
0,東京,A店,100
1,東京,B店,150
2,大阪,A店,200
3,大阪,B店,180



▼行に2階層：MultiIndex


売上
県  店舗     
東京 A店  100
   B店  150
大阪 A店  200
   B店  180

index.names: ['県', '店舗']


県   店舗    
東京  A店  売上    100
    B店  売上    150
大阪  A店  売上    200
    B店  売上    180
dtype: int64

## 2. ピボット → stack の往復
`pivot()` で列側に階層（または単層）を作ってから、`stack()` で下へ積む → `unstack()` で横へ広げる。


In [5]:
sales = pd.DataFrame({
    "県": ["東京","東京","大阪","大阪"],
    "月": ["1月","2月","1月","2月"],
    "売上": [100,150,200,250]
})

pivot = sales.pivot(index="県", columns="月", values="売上")
print("▼ピボット（横持ち）")
display(pivot)

print("\n▼stack() で列(=月)を行へ移す → Series（県×月のMultiIndex）")
st = pivot.stack()
display(st)

print("\n▼unstack() で行(=月)を列へ戻す")
display(st.unstack())


▼ピボット（横持ち）


月,1月,2月
県,,
大阪,200,250
東京,100,150



▼stack() で列(=月)を行へ移す → Series（県×月のMultiIndex）


県   月 
大阪  1月    200
    2月    250
東京  1月    100
    2月    150
dtype: int64


▼unstack() で行(=月)を列へ戻す


月,1月,2月
県,,
大阪,200,250
東京,100,150


## 3. `level` を指定して狙い撃ち（複数階層の列/行）
列や行に**複数階層**があるときは、`stack(level=...)` / `unstack(level=...)` でどの階層を動かすか指定できます。


In [6]:
# 列に2階層（項目×月）を持つ例
df = pd.DataFrame({
    ("売上","1月"): [100, 200],
    ("売上","2月"): [150, 250],
    ("利益","1月"): [ 20,  40],
    ("利益","2月"): [ 25,  45],
}, index=pd.Index(["東京","大阪"], name="県"))
df.columns = pd.MultiIndex.from_tuples(df.columns, names=["項目","月"])

print("▼列MultiIndex（項目×月）")
display(df)

print("\n▼stack(level='月'): 列の内側の '月' を行へ移す")
st_month = df.stack(level="月")  # 行は (県, 月) 階層、列は 項目
display(st_month)

print("\n▼更に stack(level='項目') で完全縦持ち（Series）へ")
st_all = df.stack(level="項目").stack(level="月")
display(st_all)

print("\n▼unstack(level='項目') で一段戻す")
display(st_all.unstack(level="項目"))


▼列MultiIndex（項目×月）


項目   売上       利益    
月    1月   2月  1月  2月
県                   
東京  100  150  20  25
大阪  200  250  40  45


▼stack(level='月'): 列の内側の '月' を行へ移す


C:\Users\kawor\AppData\Local\Temp\ipykernel_24496\2848785625.py:14: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  st_month = df.stack(level="月")  # 行は (県, 月) 階層、列は 項目


項目      売上  利益
県  月          
東京 1月  100  20
   2月  150  25
大阪 1月  200  40
   2月  250  45


▼更に stack(level='項目') で完全縦持ち（Series）へ


C:\Users\kawor\AppData\Local\Temp\ipykernel_24496\2848785625.py:18: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  st_all = df.stack(level="項目").stack(level="月")


県   項目  月 
東京  利益  1月     20
        2月     25
    売上  1月    100
        2月    150
大阪  利益  1月     40
        2月     45
    売上  1月    200
        2月    250
dtype: int64


▼unstack(level='項目') で一段戻す


項目     利益   売上
県  月          
大阪 1月  40  200
   2月  45  250
東京 1月  20  100
   2月  25  150

## 4. 欠損の扱い：unstack は穴を作り、stack はデフォルトで穴を落とす
- `unstack()` は存在しない組み合わせに **NaN** を作ります。  
- `stack()` はデフォルト `dropna=True` なので **NaN を落として** 縦に積みます。必要なら `dropna=False`。


In [9]:
df = pd.DataFrame({
    "県": ["東京","東京","大阪"],
    "月": ["1月","2月","1月"],
    "売上": [100,150,200]
})

p = df.pivot(index="県", columns="月", values="売上")
print("▼大阪の2月が欠損のピボット")
display(p)

print("\n▼stack()（デフォルト）: 欠損は落ちる → 大阪2月は消える")
display(p.stack())

print("\n▼stack(dropna=False): 欠損を落とさず保持")
display(p.stack(dropna=False))


▼大阪の2月が欠損のピボット


月,1月,2月
県,,
大阪,200.0,NaN
東京,100.0,150.0



▼stack()（デフォルト）: 欠損は落ちる → 大阪2月は消える


県   月 
大阪  1月    200.0
東京  1月    100.0
    2月    150.0
dtype: float64


▼stack(dropna=False): 欠損を落とさず保持


C:\Users\kawor\AppData\Local\Temp\ipykernel_24496\217003507.py:15: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  display(p.stack(dropna=False))


県   月 
大阪  1月    200.0
    2月      NaN
東京  1月    100.0
    2月    150.0
dtype: float64

## 5. `swaplevel` と `sort_index`：階層の入れ替え＆並べ替え
複雑な変形の後は階層順が直感とズレることがあります。`swaplevel()` と `sort_index()` で整えるのが定石。


In [10]:
df = pd.DataFrame({
    "県": ["東京","東京","大阪","大阪"],
    "店舗": ["A店","B店","A店","B店"],
    "月": ["1月","1月","2月","2月"],
    "売上": [100,150,230,250]
})

pvt = df.pivot_table(index=["県","店舗"], columns="月", values="売上", aggfunc="sum")
print("▼店舗が第2階層の行MultiIndex + 列は月")
display(pvt)

print("\n▼unstack() で月を列→行へ移動し完全縦持ち（Series）")
s = pvt.unstack()
display(s)

print("\n▼swaplevel で (県, 店舗, 月) の順番に整理 → sort_index で見やすく")
s2 = s.swaplevel(0, -1)
display(s2.sort_index())


▼店舗が第2階層の行MultiIndex + 列は月


月         1月     2月
県  店舗              
大阪 A店    NaN  230.0
   B店    NaN  250.0
東京 A店  100.0    NaN
   B店  150.0    NaN


▼unstack() で月を列→行へ移動し完全縦持ち（Series）


月      1月            2月       
店舗     A店     B店     A店     B店
県                             
大阪    NaN    NaN  230.0  250.0
東京  100.0  150.0    NaN    NaN


▼swaplevel で (県, 店舗, 月) の順番に整理 → sort_index で見やすく


TypeError: Can only swap levels on a hierarchical axis.

## 6. 現場寄りパターン：`pivot_table` × `unstack()` × `fillna()`
重複がある現場データには `pivot()` ではなく `pivot_table(aggfunc='sum')` を使い、
`unstack()` 後に `fillna(0)` で穴埋め、`astype(int)` でスッキリ出力。


In [11]:
rng = pd.date_range("2025-10-01", periods=6, freq="D")
data = pd.DataFrame({
    "県": np.random.choice(["東京","大阪"], size=20),
    "店舗": np.random.choice(["A店","B店"], size=20),
    "日付": np.random.choice(rng, size=20),
    "売上": np.random.randint(80, 260, size=20),
})
display(data.head())

# 県×店舗×日付 で合計 → 日付を列に横展開
p = data.pivot_table(index=["県","店舗"], columns="日付", values="売上", aggfunc="sum")
print("▼unstackの出番が多い現場パターン（横展開）")
wide = p.fillna(0).astype(int)
display(wide)

print("\n▼さらに分析用に縦化：stack() → tidy")
tidy = wide.stack().rename("売上").reset_index()
display(tidy.head())


,県,店舗,日付,売上
0,大阪,B店,2025-10-02,93
1,大阪,A店,2025-10-04,129
2,東京,B店,2025-10-06,219
3,大阪,A店,2025-10-02,116
4,東京,A店,2025-10-04,207


▼unstackの出番が多い現場パターン（横展開）


日付     2025-10-01  2025-10-02  2025-10-03  2025-10-04  2025-10-06
県  店舗                                                            
大阪 A店          84         303         110         315         171
   B店           0          93         359         388           0
東京 A店           0         228           0         592           0
   B店           0           0           0         338         219


▼さらに分析用に縦化：stack() → tidy


,県,店舗,日付,売上
0,大阪,A店,2025-10-01,84
1,大阪,A店,2025-10-02,303
2,大阪,A店,2025-10-03,110
3,大阪,A店,2025-10-04,315
4,大阪,A店,2025-10-06,171


## 7. クックブック：「こういう時はこう書く」
コピペで使える最短パターン集です。


In [ ]:
# 7-1) ピボットして縦→横
# df.pivot(index="県", columns="月", values="売上")

# 7-2) 横→縦（列名を下に積む）
# wide_df.stack()

# 7-3) 横→縦（複数階層の特定レベルだけ積む）
# df.stack(level="月")

# 7-4) 縦→横（最内レベルを列に）
# tall.unstack()

# 7-5) pivot だと重複でエラー → pivot_table で集計
# df.pivot_table(index=[...], columns=[...], values="売上", aggfunc="sum")

# 7-6) 欠損を残したまま積む（監査/検証など）
# wide.stack(dropna=False)

# 7-7) 穴を0で埋めてレポート用に見やすく
# wide = pivot_table.fillna(0).astype(int)

# 7-8) 階層順が気持ち悪い→入れ替え＆整列
# s.swaplevel(0, -1).sort_index()

# 7-9) 列MultiIndexをフラット化（Excel納品など）
# df.columns = ['_'.join(map(str, c)).strip() if isinstance(c, tuple) else c for c in df.columns]

# 7-10) tidy化（列名を変数列へ）: melt
# pd.melt(wide_df.reset_index(), id_vars=[...], var_name="変数名", value_name="値")


---
### メモ
- **基本原理**: `stack()` は「列→行」、`unstack()` は「行→列」。  
- **MultiIndex とセット**で覚えると一気に理解が進みます。  
- 「結果の階層順が気になる」→ `swaplevel` / `sort_index`。  
- 「重複がある」→ `pivot_table(aggfunc=...)` を選ぶ。

必要なら、このノートに**練習問題セル**も追加できます（採点用の期待形付き）。
